# AFFEC Multimodal Emotion Classification - Demo Analysis

This notebook demonstrates the AFFEC analysis workflow:
1. **Download** dataset from Zenodo (record 14794876)
2. **Shallow demo** analysis with sample data
3. **Call comprehensive** analysis via Python modules

**Key References:**
- Dataset: https://zenodo.org/records/14794876
- Documentation: See `docs/` for bias analysis, methodology, statistical rigor
- Modules: See `affec/` for data loading, feature extraction, model training

## 1. Set Up Repository Paths and Dependencies

In [ ]:
# Initialize data loader
loader = AFFECDataLoader(data_dir=str(DATA_DIR))

print("Loading AFFEC dataset...")
print("-" * 50)

# Load participants metadata
try:
    participants = loader.load_participants()
    if participants is not None:
        print(f"✓ Participants: {len(participants)} records")
        print(f"  Columns: {list(participants.columns)[:5]}...")
        print(f"\n  Demographic summary (see docs/DATASET_BIAS.md):")
        if 'gender' in participants.columns:
            print(f"  Gender: {participants['gender'].value_counts().to_dict()}")
        if 'age' in participants.columns:
            print(f"  Age range: {participants['age'].min():.0f}-{participants['age'].max():.0f}")
except Exception as e:
    print(f"⚠ Could not load participants: {e}")

# Sample trial data
try:
    participant_dirs = sorted([p for p in DATA_DIR.glob('sub-*') if p.is_dir()])
    sample_participant = participant_dirs[0].name if participant_dirs else None
    if sample_participant:
        print(f"\nLoading sample participant: {sample_participant}")
        trial = loader.load_au_data(sample_participant, run=0)
        if trial is not None:
            print(f"✓ Action Unit data: {len(trial)} frames")
            print(f"  Columns: {list(trial.columns[:5])}")
            print(f"  Action Units: {[c for c in trial.columns if 'AU' in c][:5]}...")
except Exception as e:
    print(f"⚠ Could not load AU data: {e}")

print("\nDataset structure:")
print(f"- BIDS-compliant directory structure")
print(f"- 72 participants × 4 runs = 288 sessions")
print(f"- ~20+ trials per session")
print(f"- Modalities: AU, GSR, EEG, eye-tracking, personality")
print(f"- Targets: perceived/felt × arousal/valence (1-9 continuous, to be discretized)")

## 2. Download and Verify the Zenodo Dataset

In [ ]:
# Initialize Zenodo downloader
print("Initializing Zenodo dataset downloader...")
print(f"Target record: {Config.ZENODO_RECORD_ID}")

downloader = ZenodoDataset(output_dir=str(DATA_DIR))

# Check if dataset already exists
dataset_dir = DATA_DIR / "AFFEC"
if dataset_dir.exists() and (dataset_dir / "participants.tsv").exists():
    print("✓ Dataset already downloaded locally")
else:
    print("Downloading dataset from Zenodo (this may take a few minutes)...")
    # Note: Actual download happens via AFFECDataLoader
    # For demo, we'll skip if dataset doesn't exist
    print("⚠ Dataset not found. Run: python scripts/download_data.py")
    print("  Or set DATA_DIR to existing AFFEC dataset path")


## 3. Load Files and Inspect Dataset Structure

In [ ]:
# Initialize data loader
loader = AFFECDataLoader(data_dir=str(DATA_DIR))

print("Loading AFFEC dataset...")
print("-" * 50)

# Load participants metadata
try:
    participants = loader.load_participants()
    print(f"✓ Participants: {len(participants)} records")
    print(f"  Columns: {list(participants.columns)[:5]}...")
    print(f"\n  Demographic summary (see docs/DATASET_BIAS.md):")
    print(f"  Gender: {participants['gender'].value_counts().to_dict() if 'gender' in participants.columns else 'N/A'}")
    print(f"  Age range: {participants['age'].min():.0f}-{participants['age'].max():.0f}" if 'age' in participants.columns else "")
except Exception as e:
    print(f"⚠ Could not load participants: {e}")

# Sample trial data
try:
    sample_participant = loader.participant_dirs[0].name if loader.participant_dirs else None
    if sample_participant:
        print(f"\nLoading sample participant: {sample_participant}")
        trial = loader.load_au_data(sample_participant, run=0)
        print(f"✓ Action Unit data: {len(trial)} frames")
        print(f"  Columns: {list(trial.columns[:5])}")
        print(f"  Action Units: {[c for c in trial.columns if 'AU' in c][:5]}...")
except Exception as e:
    print(f"⚠ Could not load AU data: {e}")

print("\nDataset structure:")
print(f"- BIDS-compliant directory structure")
print(f"- 72 participants × 4 runs = 288 sessions")
print(f"- ~20+ trials per session")
print(f"- Modalities: AU, GSR, EEG, eye-tracking, personality")
print(f"- Targets: perceived/felt × arousal/valence (1-9 continuous, to be discretized)")


## 4. Run a Shallow Demo Analysis in the Notebook

For demo purposes, we'll:
1. Extract features from Action Unit data on one participant
2. Show baseline statistics
3. Demonstrate the feature extraction workflow

In [ ]:
# Demo feature extraction on a small sample if data exists
try:
    if 'sample_participant' in locals() and sample_participant:
        trial_data = loader.load_au_data(sample_participant, run=0)
        feature_extractor = MultimodalFeatureExtractor(use_modalities=Config.USE_MODALITIES)
        
        # Build a minimal demo trial dictionary compatible with the extractor
        demo_trial = {
            'participant': sample_participant,
            'run': 0,
            'stimulus_emotion': 'demo',
            'perceived_arousal': np.nan,
            'perceived_valence': np.nan,
            'felt_arousal': np.nan,
            'felt_valence': np.nan,
            'au_data': trial_data,
            'personality': {'O': np.nan, 'C': np.nan, 'E': np.nan, 'A': np.nan, 'N': np.nan}
        }
        
        demo_features = feature_extractor.extract_trial_features(demo_trial)
        demo_features_df = pd.DataFrame([demo_features])
        print("✓ Extracted demo features")
        print(f"  Feature count: {demo_features_df.shape[1]}")
        display(demo_features_df.head())
    else:
        print("No sample data available; demo analysis skipped.")
except Exception as e:
    print(f"⚠ Demo analysis could not run: {e}")


## 5. Call Full Analysis from Python Modules

In [ ]:
# Run full analysis through Python modules
try:
    config = Config()
    logger = ExperimentLogger(log_dir=str(LOGS_DIR), experiment_name="affec_demo")
    logger.log_config(config.to_dict())
    
    print("Full analysis should be executed via the Python modules in affec/:")
    print("- affec/data/loader.py: dataset download and loading")
    print("- affec/features/extractor.py: feature extraction")
    print("- affec/models/baseline.py: 5-fold baseline training")
    print("- affec/utils/logging.py: experiment tracking")
    
    logger.add_decision(
        title="Notebook kept shallow",
        rationale="Heavy analysis is delegated to modular Python code for reproducibility and reuse",
        evidence="Repository now contains reusable affec/ modules"
    )
    logger.add_next_action(
        action="Run comprehensive training script against the Zenodo dataset",
        priority="high"
    )
    saved = logger.save()
    print(f"✓ Logged experiment metadata to {saved}")
except Exception as e:
    print(f"⚠ Could not initialize full analysis logger: {e}")


## 6. Compute Core Summary Statistics

In [ ]:
# Summary statistics on available data
try:
    if 'participants' in locals() and not participants.empty:
        print("Participants summary:")
        display(participants.describe(include='all').T.head(15))
        
        if 'gender' in participants.columns:
            print("\nGender counts:")
            display(participants['gender'].value_counts(dropna=False))
        
        numeric_cols = participants.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_cols:
            print("\nNumeric correlations:")
            corr = participants[numeric_cols].corr(numeric_only=True)
            display(corr.head())
    else:
        print("No participants data available for summary statistics.")
except Exception as e:
    print(f"⚠ Summary statistics unavailable: {e}")


## 7. Create Basic Visualizations

In [ ]:
# Basic visualizations
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    plt.style.use('seaborn-v0_8')
    
    if 'participants' in locals() and 'gender' in participants.columns:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        
        participants['gender'].value_counts().plot(kind='bar', ax=ax[0], color='steelblue')
        ax[0].set_title('Participant Gender Distribution')
        ax[0].set_xlabel('Gender')
        ax[0].set_ylabel('Count')
        
        if 'age' in participants.columns:
            participants['age'].hist(ax=ax[1], bins=10, color='darkorange', edgecolor='black')
            ax[1].set_title('Age Distribution')
            ax[1].set_xlabel('Age')
            ax[1].set_ylabel('Count')
        
        plt.tight_layout()
        plt.show()
    
    if 'participants' in locals():
        numeric_cols = participants.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) > 1:
            plt.figure(figsize=(8, 6))
            sns.heatmap(participants[numeric_cols].corr(numeric_only=True), annot=False, cmap='coolwarm')
            plt.title('Correlation Heatmap')
            plt.tight_layout()
            plt.show()
except Exception as e:
    print(f"⚠ Visualization skipped: {e}")


## 8. Save Tables, Figures, and Intermediate Artifacts

In [ ]:
# Save artifacts for reproducibility
try:
    summary_path = PROCESSED_DIR / "participants_summary.csv"
    if 'participants' in locals():
        participants.to_csv(summary_path, index=False)
        print(f"✓ Saved participants table to {summary_path}")
    
    if 'demo_features_df' in locals():
        features_path = PROCESSED_DIR / "demo_features.csv"
        demo_features_df.to_csv(features_path, index=False)
        print(f"✓ Saved demo features to {features_path}")
    
    print(f"Output directories ready:\n- {PROCESSED_DIR}\n- {FIGURES_DIR}\n- {LOGS_DIR}")
except Exception as e:
    print(f"⚠ Could not save artifacts: {e}")


In [ ]:
print("=" * 60)
print("SHALLOW DEMO ANALYSIS")
print("=" * 60)

# Demo: Extract features from sample trials
print("\n1. Feature Extraction Demo")
print("-" * 40)

extractor = MultimodalFeatureExtractor(
    use_modalities={'au': True, 'gsr': False, 'eye': False, 'personality': True}
)

# Create synthetic demo trial (in real scenario, would load from data)
demo_trial = {
    'participant': 'sub-demo',
    'run': 0,
    'stimulus_emotion': 'happy',
    'perceived_arousal': 7,
    'perceived_valence': 8,
    'felt_arousal': 6,
    'felt_valence': 7,
    'au_data': pd.DataFrame({
        'AU01_r': np.random.uniform(0, 1, 30),
        'AU02_r': np.random.uniform(0, 0.5, 30),
        'AU06_r': np.random.uniform(0, 1, 30),
        'AU12_r': np.random.uniform(0.5, 1, 30),
        'onset': np.linspace(0, 3, 30)
    }),
    'personality': {
        'openness': 5, 'conscientiousness': 6, 'extraversion': 7,
        'agreeableness': 5, 'neuroticism': 4
    }
}

features = extractor.extract_trial_features(demo_trial)
print(f"✓ Extracted {len(features)} features from trial")
print(f"\n  Sample features:")
for key in list(features.keys())[:8]:
    print(f"  - {key}: {features[key]:.3f}" if isinstance(features[key], (int, float)) else f"  - {key}: {features[key]}")

# Demo: Show modality configuration options
print("\n2. Modality Configuration")
print("-" * 40)
print(f"Available modalities: {list(MODALITIES.keys())}")
for mod, desc in MODALITIES.items():
    print(f"  - {mod}: {desc}")
    
print(f"\nTarget variables: {list(TARGETS.keys())}")
for target, desc in TARGETS.items():
    print(f"  - {target}: {desc}")

print("\n3. Configuration Summary (from affec.utils.config)")
print("-" * 40)
config = Config()
print(f"  N_FOLDS: {config.N_FOLDS} (per STATISTICAL_ANALYSIS.md)")
print(f"  AU_WINDOW: {config.AU_WINDOW}s (per METHODOLOGICAL_CHOICES.md)")
print(f"  EMOTION_BINS: {config.EMOTION_BINS} (Low/Medium/High)")
print(f"  Data directory: {config.DATA_DIR}")
